# Checkpoint 2 — offline CCS / PA-CCS / behavior

Запускается в новом процессе после создания model artifacts. Основные модели не загружаются. Compatible CCS probes переиспользуются; при изменении CCS config обучение выполняется только на сохранённых hidden states.

In [ ]:
from pathlib import Path
import sys

sys.dont_write_bytecode = True
PROJECT = next(
    candidate
    for base in (Path.cwd(), *Path.cwd().parents)
    for candidate in (base, base / "latent-behavior-alignment")
    if (candidate / "notebooks/checkpoint2/artifacts.py").is_file()
)
MODULE_DIR = PROJECT / "notebooks/checkpoint2"
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))
from artifacts import MODELS, load_dataset, output_root

DATASET = "mixed"
ARTIFACTS = output_root(DATASET) / "artifacts"
data = load_dataset(dataset=DATASET)
SELECTED_MODELS = list(MODELS)  # Or ["gemma-2-2b"] for the first run.
print("Dataset:", len(data), "train:", len(data.train_idx), "test:", len(data.test_idx))


In [ ]:
from offline import analyze, behavioral_scores, pa_probabilities
from artifacts import load_cache

THRESHOLD = 0.5
CCS_DEVICE = "cpu"  # Small probes can also run on CUDA; no main model is loaded.
PREPROCESSING = "checkpoint1"
CCS_SEED = 0
NEPOCHS = 1500
NTRIES = 10
LAYERS = None  # All layers, including embedding layer 0.
FORCE_RETRAIN = False
RUN_NAME = "checkpoint1_compatible"  # Change for separate experiment exports.
OUTPUT_DIR = output_root(DATASET) / "analysis" / RUN_NAME


## Переобучить или восстановить probes и рассчитать результаты

По умолчанию исходный split 0.2/71. `accuracy` — исходная sign-invariant CCS accuracy; `latent_vs_true` использует ориентацию, выбранную по train. PA-CCS сохраняет original pairing, включая counterpart из train.

In [ ]:
for name in SELECTED_MODELS:
    scores, metrics = analyze(
        ARTIFACTS / name, MODELS[name], data,
        threshold=THRESHOLD, device=CCS_DEVICE, preprocessing=PREPROCESSING,
        seed=CCS_SEED, nepochs=NEPOCHS, ntries=NTRIES,
        layers=LAYERS, force_retrain=FORCE_RETRAIN,
    )
    destination = OUTPUT_DIR / name
    destination.mkdir(parents=True, exist_ok=True)
    scores.to_csv(destination / "scores.csv", index=False)
    metrics.to_csv(destination / "layer_metrics.csv", index=False)
    print(name, "→", destination)
    display(metrics)


In [ ]:
# Standard run: export the same summaries and CPU validation as run_all.py.
if LAYERS is None and RUN_NAME == "checkpoint1_compatible":
    from validation import export_validation
    validation = export_validation(
        output_root(DATASET), data, {name: MODELS[name] for name in SELECTED_MODELS},
        threshold=THRESHOLD, preprocessing=PREPROCESSING,
        seed=CCS_SEED, nepochs=NEPOCHS, ntries=NTRIES,
    )
    print("Validation:", validation["validation"])


## Изменить только behavioral threshold

Ни обучения CCS, ни model inference. Результат содержит все statements; test можно выбрать по `data.test_idx`.

In [ ]:
name = SELECTED_MODELS[0]
cache = load_cache(ARTIFACTS / name, MODELS[name], data)
behavior = behavioral_scores(cache, threshold=0.7)
display(behavior.head())
del cache


## PA-CCS probabilities из существующего probe

Вычисляются в памяти, без обучения. Сохранённые split и preprocessing берутся из derived cache.

In [ ]:
layer = 0 if LAYERS is None else LAYERS[0]
probabilities = pa_probabilities(ARTIFACTS / name, MODELS[name], data, layer, device=CCS_DEVICE)
display(probabilities.head())
